# Build your own widget

cadetgui's widgets are made from a small kit:

- `cadetgui.widgets.Widget`: the base class. Set `self.root` to an ipywidgets box, call `self._notify(...)` when the value changes; others subscribe with `add_listener`.
- `cadetgui.widgets.ui`: layout helpers (`panel`, `section`, `row`, `note`, `collapsible`, `CodeView`) and HTML snippets (`status_html`, `table_html`).
- `cadetgui.widgets.elements`: inputs and charts (`FloatField`, `ChoiceField`, `EventTimelineChart`, `ChromatogramChart`, ...).

This notebook builds a widget that sets the cycle time of a process and shows its events.

## A process with two events

A tracer pulse: the inlet feeds the tracer from 0 s (`load`) and stops at 60 s (`wash`).

In [ ]:
from CADETProcess.processModel import (
    ComponentSystem, FlowSheet, Inlet, LumpedRateModelWithoutPores, Outlet, Process,
)

component_system = ComponentSystem(["Tracer"])

inlet = Inlet(component_system, "inlet")
inlet.flow_rate = 1e-7

column = LumpedRateModelWithoutPores(component_system, "column")
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7

outlet = Outlet(component_system, "outlet")

flow_sheet = FlowSheet(component_system, "flow_sheet")
for unit in (inlet, column, outlet):
    flow_sheet.add_unit(unit)
flow_sheet.add_connection(inlet, column)
flow_sheet.add_connection(column, outlet)

process = Process(flow_sheet, "pulse")
process.cycle_time = 600
process.add_event("load", "flow_sheet.inlet.c", 1.0, 0)
_ = process.add_event("wash", "flow_sheet.inlet.c", 0.0, 60)

## The widget

`CycleTimeWidget` changes the cycle time. With *scale with cycle time*, the event times are scaled by the same factor. The timeline is drawn with `show_timeline`.

In [ ]:
import html

import ipywidgets as W

from cadetgui.widgets import Widget
from cadetgui.widgets.elements import ChoiceField, EventTimelineChart, FloatField
from cadetgui.widgets.process_model.timeline import show_timeline
from cadetgui.widgets.ui import (
    CodeView, collapsible, note, panel, row, section, status_html, table_html,
)


class CycleTimeWidget(Widget):
    """Set a process's cycle time; the event times scale with it or stay put."""

    def __init__(self, process):
        super().__init__()
        self.process = process
        self._cycle_time = process.cycle_time
        self._event_times = {event.name: event.time for event in process.events}

        self.cycle_time = FloatField(
            label="cycle_time", value=process.cycle_time, min=1, units=r"\mathrm{s}"
        )
        self.mode = ChoiceField(
            label="Event times:", options=[("scale with cycle time", True), ("keep", False)]
        )
        self.status = W.HTML()
        self.chart = EventTimelineChart()
        self.table = W.HTML()
        self.code_view = CodeView(self.code)

        self.root = panel(
            f"Cycle time · {html.escape(process.name)}",
            note("Change the cycle time; the events move with it or keep their times."),
            section([row(self.cycle_time, self.mode), self.status], title_html="Settings"),
            self.chart,
            collapsible("Events", self.table),
            self.code_view.root,
        )
        self.cycle_time.observe(lambda _change: self.apply(), names="value")
        self.mode.observe(lambda _change: self.apply(), names="selected_index")
        self.apply()

    def apply(self):
        factor = self.cycle_time.value / self._cycle_time if self.mode.value else 1.0
        self.process.cycle_time = self.cycle_time.value
        for event in self.process.events:
            event.time = self._event_times[event.name] * factor

        late = [e.name for e in self.process.events if e.time >= self.process.cycle_time]
        self.status.value = (
            status_html("warn", f"After the cycle end: {', '.join(late)}")
            if late else status_html("ok", f"Cycle time {self.process.cycle_time:g} s")
        )
        show_timeline(self.chart, self.process, self.process.component_system.names)
        self.table.value = table_html(
            ["Event", "Time / s", "Parameter", "State"],
            [[html.escape(e.name), f"{e.time:g}", e.parameter_path, f"{e.state}"]
             for e in self.process.events],
        )
        self.code_view.refresh()
        self._notify(self.process)

    def code(self):
        lines = [f"process.cycle_time = {self.process.cycle_time:g}"]
        lines += [f"process.{e.name}.time = {e.time:g}" for e in self.process.events]
        return "\n".join(lines)

## Use it

A widget shows itself as the last line of a cell.

In [ ]:
cycle = CycleTimeWidget(process)
cycle

## Listen to changes

`add_listener` calls a function after every change; here it prints the event times. Setting a field in code works like typing into it.

In [ ]:
cycle.add_listener(lambda p: print({e.name: e.time for e in p.events}))
cycle.cycle_time.value = 1200

In [ ]:
print(cycle.code())